## [ICIT] Original

Code adapted from ([link](https://github.com/wanderfernandesjunior/mestrado))

In [2]:
# Artifício para calcular tempo total do notebook Jupyter
from datetime import datetime
start_time = datetime.now()

import pandas as pd
import numpy as np
import seaborn as sns
import logging
import warnings
import sys
from math import ceil
from matplotlib import pyplot as plt
from time import time
from pathlib import Path

from tsfresh.feature_extraction import extract_features
from tsfresh.utilities.dataframe_functions import impute
from tsfresh.feature_extraction import MinimalFCParameters

from sklearn.svm import OneClassSVM
from sklearn.covariance import EllipticEnvelope
from sklearn.neighbors import LocalOutlierFactor
from sklearn.ensemble import IsolationForest
from sklearn.dummy import DummyClassifier
from sklearn.metrics import confusion_matrix
from sklearn import preprocessing
from sklearn.metrics import precision_recall_fscore_support
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score
from sklearn.model_selection import ParameterGrid

logging.getLogger('tsfresh').setLevel(logging.ERROR)
warnings.simplefilter(action='ignore')

%matplotlib inline
%config InlineBackend.figure_format = 'png'

data_path = Path('./', 'data')

random_state = 1
np.random.seed(random_state)

events_names = {0: 'Normal',
                1: 'Aumento Abrupto de BSW',
                2: 'Fechamento Espúrio de DHSV',
                3: 'Intermitência Severa',
                4: 'Instabilidade de Fluxo',
                5: 'Perda Rápida de Produtividade',
                6: 'Restrição Rápida em CKP',
                7: 'Incrustação em CKP',
                8: 'Hidrato em Linha de Produção'
                }

vars = ['P-PDG',
        'P-TPT',
        'T-TPT',
        'P-MON-CKP',
        'T-JUS-CKP',
        'P-JUS-CKGL',
        'T-JUS-CKGL',
        'QGL']

columns = ['timestamp'] + vars + ['class']

normal_class_code = 0
abnormal_classes_codes = [1, 2, 5, 6, 7, 8]

sample_size = 3*60  # Nas observações = segundos
min_normal_period_size = 20*60  # Nas observações = segundos
split_range = 0.6  # Porcentagem de separação entre treino/teste
max_samples_per_period = 15  # limitação por 'segurança'

df_fc_p = MinimalFCParameters()  # Ver documentação da biblioteca tsfresh
# Uso de .pop(..., None) para evitar KeyError caso a versão do tsfresh
# não contenha mais essas chaves por padrão
df_fc_p.pop('absolute_maximum', None)  # Remove para compatibilidade com o artigo
df_fc_p.pop('sum_values', None)  # Remove feature inapropriada
df_fc_p.pop('length', None)  # Remove feature inapropriada

max_nan_percent = 0.1  # Para seleção de variáveis úteis
std_vars_min = 0.01  # Para seleção de variáveis úteis

clfs = {}  # Dicionário para lista de classificadores a serem experimentados
disable_progressbar = True  # Para menos saídas no notebook


def class_and_file_generator(data_path, real=False, simulated=False, drawn=False):
    """Gerador de lista contendo número da classe e caminho do arquivo de acordo com a fonte da instância."""
    for class_path in data_path.iterdir():
        if class_path.is_dir():
            class_code = int(class_path.stem)
            for instance_path in class_path.iterdir():
                if (instance_path.suffix == '.csv'):
                    if (simulated and instance_path.stem.startswith('SIMULATED')) or \
                       (drawn and instance_path.stem.startswith('DRAWN')) or \
                       (real and (not instance_path.stem.startswith('SIMULATED')) and
                        (not instance_path.stem.startswith('DRAWN'))):
                        yield class_code, instance_path


def load_instance(instance_path):
    """Função que carrega cada instância individualmente"""
    try:
        well, instance_id = instance_path.stem.split('_')
        df = pd.read_csv(instance_path, sep=',', header=0)
        assert (df.columns == columns).all(), \
            f'Colunas inválidas no arquivo {str(instance_path)}: {str(df.columns.tolist())}'
        return df
    except Exception as e:
        raise Exception(f'Erro ao ler arquivo {instance_path}: {e}')


def extract_samples(df, class_code):
    ols = list(df['class'])
    set_ols = set()
    for ol in ols:
        if ol in set_ols or np.isnan(ol):
            continue
        set_ols.add(int(ol))

    df_vars = df.drop('class', axis=1).fillna(0)

    df_samples_train = pd.DataFrame()
    df_samples_test = pd.DataFrame()
    y_train = []
    y_test = []

    f_idx = ols.index(normal_class_code)
    l_idx = len(ols)-1-ols[::-1].index(normal_class_code)
    max_samples_normal = l_idx-f_idx+1-sample_size

    if (max_samples_normal) > 0:
        num_normal_samples = min(max_samples_per_period, max_samples_normal)
        num_train_samples = int(split_range*num_normal_samples)
        num_test_samples = num_normal_samples - num_train_samples
    else:
        num_train_samples = 0
        num_test_samples = 0

    transient_code = class_code + 100
    if transient_code in set_ols:
        f_idx = ols.index(transient_code)
        if f_idx-(sample_size-1) > 0:
            f_idx = f_idx-(sample_size-1)
        else:
            f_idx = 0
        l_idx = len(ols)-1-ols[::-1].index(transient_code)
        max_transient_samples = l_idx-f_idx+1-sample_size
    else:
        max_transient_samples = 0

    if class_code in set_ols:
        f_idx = ols.index(class_code)
        if f_idx-(sample_size-1) > 0:
            f_idx = f_idx-(sample_size-1)
        else:
            f_idx = 0
        l_idx = len(ols)-1-ols[::-1].index(class_code)
        if l_idx+(sample_size-1) < len(ols)-1:
            l_idx = l_idx+(sample_size-1)
        else:
            l_idx = len(ols)-1
        max_in_regime_samples = l_idx-f_idx+1-sample_size
    else:
        max_in_regime_samples = 0

    num_transient_samples = ceil(num_test_samples/2)
    num_in_regime_samples = num_test_samples - num_transient_samples

    if (max_transient_samples >= num_transient_samples) and \
       (max_in_regime_samples < num_in_regime_samples):
        num_in_regime_samples = max_in_regime_samples
        num_transient_samples = min(num_test_samples-num_in_regime_samples, max_transient_samples)
    elif (max_transient_samples < num_transient_samples) and \
         (max_in_regime_samples >= num_in_regime_samples):
        num_transient_samples = max_transient_samples
        num_in_regime_samples = min(num_test_samples-num_transient_samples, max_in_regime_samples)
    elif (max_transient_samples < num_transient_samples) and \
         (max_in_regime_samples < num_in_regime_samples):
        num_transient_samples = max_transient_samples
        num_in_regime_samples = max_in_regime_samples

    num_test_samples = num_transient_samples+num_in_regime_samples

    f_idx = ols.index(normal_class_code)
    l_idx = len(ols)-1-ols[::-1].index(normal_class_code)

    if (num_normal_samples) > 0:
        if num_normal_samples == max_samples_normal:
            step_max = 1
        else:
            step_max = (max_samples_normal-1) // (max_samples_per_period-1)
        step_wanted = sample_size
        step = min(step_wanted, step_max)

        sample_id = 0
        for idx in range(num_train_samples):
            f_idx_c = l_idx-sample_size+1-(num_normal_samples-1-idx)*step
            l_idx_c = f_idx_c+sample_size
            df_sample = df_vars.iloc[f_idx_c:l_idx_c, :]
            df_sample.insert(loc=0, column='id', value=sample_id)
            df_samples_train = pd.concat([df_samples_train, df_sample])
            y_train.append(normal_class_code)
            sample_id += 1

        sample_id = 0
        for idx in range(num_train_samples, num_train_samples+num_test_samples):
            f_idx_c = l_idx-sample_size+1-(num_normal_samples-1-idx)*step
            l_idx_c = f_idx_c+sample_size
            df_sample = df_vars.iloc[f_idx_c:l_idx_c, :]
            df_sample.insert(loc=0, column='id', value=sample_id)
            df_samples_test = pd.concat([df_samples_test, df_sample])
            y_test.append(normal_class_code)
            sample_id += 1

    if (num_transient_samples) > 0:
        if num_transient_samples == max_transient_samples:
            step_max = 1
        else:
            step_max = (max_transient_samples-1) // (max_samples_per_period-1)
        step_wanted = np.inf
        step = min(step_wanted, step_max)

        f_idx = ols.index(transient_code)
        if f_idx-(sample_size-1) > 0:
            f_idx = f_idx-(sample_size-1)
        else:
            f_idx = 0
        l_idx = len(ols)-1-ols[::-1].index(transient_code)

        for idx in range(num_transient_samples):
            f_idx_c = f_idx+idx*step
            l_idx_c = f_idx_c+sample_size
            df_sample = df_vars.iloc[f_idx_c:l_idx_c, :]
            df_sample.insert(loc=0, column='id', value=sample_id)
            df_samples_test = pd.concat([df_samples_test, df_sample])
            y_test.append(transient_code)
            sample_id += 1

    if (num_in_regime_samples) > 0:
        if num_in_regime_samples == max_in_regime_samples:
            step_max = 1
        else:
            step_max = (max_in_regime_samples-1) // (max_samples_per_period-1)
        step_wanted = sample_size
        step = min(step_wanted, step_max)

        f_idx = ols.index(class_code)
        if f_idx-(sample_size-1) > 0:
            f_idx = f_idx-(sample_size-1)
        else:
            f_idx = 0
        l_idx = len(ols)-1-ols[::-1].index(class_code)
        if l_idx+(sample_size-1) < len(ols)-1:
            l_idx = l_idx+(sample_size-1)
        else:
            l_idx = len(ols)-1

        for idx in range(num_in_regime_samples):
            f_idx_c = f_idx+idx*step
            l_idx_c = f_idx_c+sample_size
            df_sample = df_vars.iloc[f_idx_c:l_idx_c, :]
            df_sample.insert(loc=0, column='id', value=sample_id)
            df_samples_test = pd.concat([df_samples_test, df_sample])
            y_test.append(class_code)
            sample_id += 1

    return df_samples_train, y_train, df_samples_test, y_test


def train_test_calc_scores(X_train, y_train, X_test, y_test, scores, clfs,
                            n_training_samples, n_test_samples):
    X_train.reset_index(inplace=True, drop=True)
    X_test.reset_index(inplace=True, drop=True)

    for clf_name, clf in clfs.items():
        t0 = time()
        clf.fit(X_train, y_train)
        t_train = time() - t0

        t0 = time()
        y_pred = clf.predict(X_test)
        t_test = time() - t0

        ret = precision_recall_fscore_support(y_test, y_pred, average='binary')
        # p, r, f1, _ = ret
        # cm = confusion_matrix(y_test, y_pred, labels=[1, -1])
        # TP = cm[0, 0]
        # FN = cm[0, 1]
        # FP = cm[1, 0]
        # TN = cm[1, 1]
        # total = TP + TN + FP + FN
        # accuracy = (TP + TN) / total if total > 0 else 0.0

        p = precision_score(y_test, y_pred, average="micro")
        r = recall_score(y_test, y_pred, average="micro")
        f1 = f1_score(y_test, y_pred, average="micro")
        accuracy = accuracy_score(y_test, y_pred)

        novo_registro = pd.DataFrame([{
            'CLASSIFICADOR': clf_name,
            'PRECISAO': p,
            'REVOGACAO': r,
            'F1': f1,
            'ACURACIA': accuracy,
            'TREINAMENTO [s]': t_train,
            'TESTE [s] ': t_test,
            'N_TRAINING_SAMPLES': n_training_samples,
            'N_TEST_SAMPLES': n_test_samples
        }])
        scores = pd.concat([scores, novo_registro], ignore_index=True)

    return scores


real_instances = pd.DataFrame(class_and_file_generator(data_path,
                                                         real=True,
                                                         simulated=False,
                                                         drawn=False),
                               columns=['class_code', 'instance_path'])

real_instances = real_instances.loc[real_instances.iloc[:, 0].isin(abnormal_classes_codes)].reset_index(drop=True)

scores = pd.DataFrame()
ignored_instances = 0
used_instances = 0

for i, row in real_instances.iterrows():
    class_code, instance_path = row
    print(f'Instância {i+1}: {instance_path}')

    df = load_instance(instance_path)

    normal_period_size = (df['class'] == float(normal_class_code)).sum()
    if normal_period_size < min_normal_period_size:
        ignored_instances += 1
        print(f'\tignorado porque normal_period_size é insuficiente para treinamento ({normal_period_size})\n')
        continue

    used_instances += 1

    ret = extract_samples(df, class_code)
    df_samples_train, y_train, df_samples_test, y_test = ret

    y_train = np.array(y_train)
    y_test = np.array(y_test)

    y_train[y_train != normal_class_code] = -1
    y_train[y_train == normal_class_code] = 1
    y_test[y_test != normal_class_code] = -1
    y_test[y_test == normal_class_code] = 1

    good_vars = np.isnan(df_samples_train[vars]).mean(axis=0) <= max_nan_percent
    std_vars = np.nanstd(df_samples_train[vars], axis=0)
    good_vars &= (std_vars > std_vars_min)
    good_vars = list(good_vars.index[good_vars])
    bad_vars = list(set(vars)-set(good_vars))

    df_samples_train.drop(columns=bad_vars, inplace=True, errors='ignore')
    df_samples_test.drop(columns=bad_vars, inplace=True, errors='ignore')

    scaler = preprocessing.StandardScaler()
    df_samples_train[good_vars] = scaler.fit_transform(df_samples_train[good_vars]).astype('float32')
    df_samples_test[good_vars] = scaler.transform(df_samples_test[good_vars]).astype('float32')

    X_train = extract_features(df_samples_train,
                                column_id='id',
                                column_sort='timestamp',
                                default_fc_parameters=df_fc_p,
                                impute_function=impute,
                                n_jobs=0,
                                disable_progressbar=disable_progressbar)
    X_train = X_train.reset_index(drop=True)

    X_test = extract_features(df_samples_test,
                               column_id='id',
                               column_sort='timestamp',
                               default_fc_parameters=df_fc_p,
                               impute_function=impute,
                               n_jobs=0,
                               disable_progressbar=disable_progressbar)
    X_test = X_test.reset_index(drop=True)

    n_training_samples = len(X_train)
    n_test_samples = len(X_test)

    clfs = {'Local Outlier Factor': LocalOutlierFactor(n_neighbors=20, algorithm='auto',
                                                        metric='euclidean', contamination="auto", novelty=True),
            'Floresta de Isolamento': IsolationForest(n_estimators=200, max_samples=0.75, contamination=0.1,
                                                        max_features=1.0, bootstrap=False,
                                                        random_state=1, verbose=0),
            'Envelope Eliptico MCD': EllipticEnvelope(contamination=0.001, assume_centered=False,
                                                       support_fraction=0.99, random_state=1),
            'One Class SVM': OneClassSVM(kernel='sigmoid', gamma=0.01, nu=0.001),
            'One Class SVM (benchmark Vargas)': OneClassSVM(kernel='sigmoid', gamma='scale', nu=0.5),
            'Floresta de Isolamento (benchmark Vargas)': IsolationForest(n_jobs=None,
                                                                          contamination=0.000001, random_state=random_state),
            'Dummy': DummyClassifier(strategy='constant', constant=1),
            'Local Outlier Factor (STD)': LocalOutlierFactor(n_neighbors=20, algorithm='auto', metric='minkowski', contamination='auto', novelty=True),
            'Envelope Eliptico MCD (STD)': EllipticEnvelope(contamination=0.1, assume_centered=False, support_fraction=None, random_state=1),

           }

    scores = train_test_calc_scores(X_train, y_train, X_test, y_test, scores, clfs,
                                     n_training_samples, n_test_samples)

print(f'Número de instâncias utilizadas: {used_instances}')
print(f'Número de instâncias ignoradas: {ignored_instances}')
print(f'Características utilizadas: {list(df_fc_p.keys())}')

scores.to_csv(r'./results/2-0_anomaly_detection_scores_reais.csv')

scores['RODADA'] = scores.groupby('CLASSIFICADOR').cumcount() + 1

metricas = ['PRECISAO', 'REVOGACAO', 'F1', 'ACURACIA', 'TREINAMENTO [s]', 'TESTE [s] ',
            'N_TRAINING_SAMPLES', 'N_TEST_SAMPLES']

sheet_names = {m: m.replace('[', '(').replace(']', ')').strip()[:31] for m in metricas}

with pd.ExcelWriter(r'./results/2-0_anomaly_detection_scores_reais.xlsx', engine='openpyxl') as writer:
    for metrica in metricas:
        pivot = scores.pivot(index='RODADA', columns='CLASSIFICADOR', values=metrica)
        pivot.columns.name = None
        pivot.index.name = 'RODADA'
        pivot.to_excel(writer, sheet_name=sheet_names[metrica])

mean_score_table = scores.groupby('CLASSIFICADOR').mean(numeric_only=True).sort_values(by=['F1'], ascending=False)
mean_score_table.to_csv(r'./results/2-0_anomaly_detection_scores_medias_reais.csv')
print(mean_score_table[['PRECISAO', 'REVOGACAO', 'F1', 'ACURACIA']])

std_score_table = scores.groupby('CLASSIFICADOR').std(numeric_only=True).sort_values(by=['F1'], ascending=True)
std_score_table.to_csv(r'./results/2-0_anomaly_detection_scores_desvios_padrao_reais.csv')
print(std_score_table)

print(f'Tempo total de execução (hh:mm:ss.ms): {datetime.now() - start_time}')


Instância 1: data\1\WELL-00001_20140124213136.csv
	ignorado porque normal_period_size é insuficiente para treinamento (959)

Instância 2: data\1\WELL-00002_20140126200050.csv
	ignorado porque normal_period_size é insuficiente para treinamento (1138)

Instância 3: data\1\WELL-00006_20170801063614.csv
Instância 4: data\1\WELL-00006_20170802123000.csv
Instância 5: data\1\WELL-00006_20180618060245.csv
Instância 6: data\2\WELL-00002_20131104014101.csv
Instância 7: data\2\WELL-00003_20141122214325.csv
Instância 8: data\2\WELL-00003_20170728150240.csv
Instância 9: data\2\WELL-00003_20180206182917.csv
	ignorado porque normal_period_size é insuficiente para treinamento (586)

Instância 10: data\2\WELL-00009_20170313160804.csv
Instância 11: data\2\WELL-00010_20171218200131.csv
Instância 12: data\2\WELL-00011_20140515110134.csv
Instância 13: data\2\WELL-00011_20140530100015.csv
	ignorado porque normal_period_size é insuficiente para treinamento (482)

Instância 14: data\2\WELL-00011_2014060623011